## Importy


In [ ]:
from utils.sinogram import show_sinogram, scan_generate_sinogram
from utils.filter import filter_sinogram
from utils.image_reconstruction import reconstruct_image, show_image
from utils.dicom import save_reconstruction_to_dicom, load_dicom_file

## Wybór zdjęcia


In [ ]:
from utils.choose_image import Choose_image
ui = Choose_image()
ui.show()

## Parametry wejściowe


In [ ]:
import math
from ipywidgets import interact, IntSlider, Text

img_matrix = ui.img_matrix
img_width = ui.img_width
img_height = ui.img_height

patient_name = None
patient_id = None
comment = None


def symuluj_tomograf(n_detectors, n_scans, angle_cov, det_span, p_name, p_id, p_comment):
    global N_detectors, N_scans, angle_coverage, distance_between_emiters 
    global patient_name, patient_id, comment
    
    patient_name = p_name if p_name.strip() else None
    patient_id = p_id if p_id.strip() else None
    comment = p_comment if p_comment.strip() else None

    N_detectors = n_detectors
    N_scans = n_scans
    angle_coverage = angle_cov
    distance_between_emiters = det_span/100*math.sqrt(pow(img_width,2)+pow(img_height,2))/N_detectors 
   
    print("--- PARAMETRY SKANU ---")
    print(f"Liczba detektorów: {n_detectors}")
    print(f"Liczba skanów: {n_scans}")
    print(f"Kąt: {angle_cov}°")
    print(f"Rozpiętość układu emiterów-detektorów: {det_span}% maksymalnego wymiaru obrazu")
    print(f"Dystans między emiterami: {distance_between_emiters:.4f} px")
    print("--- PARAMETRY PACJENTA ---")
    print(f"Pacjent: {patient_name or 'Domyślny (Kowalski^Jan)'}")
    print(f"ID: {patient_id or 'Domyślne (pacjent0)'}")
    print(f"Komentarz: {comment or 'Brak'}")
    

# Tworzenie interaktywnego panelu
interact(
    symuluj_tomograf, 
    n_detectors=IntSlider(min=90, max=720, step=10, value=180),
    n_scans=IntSlider(min=90, max=720, step=10, value=180),
    angle_cov=IntSlider(min=45, max=360, step=5, value=180),
    det_span=IntSlider(min=1, max=100, step=5, value=100),
    p_name=Text(value='', placeholder='Imię^Nazwisko', description='Pacjent:'),
    p_id=Text(value='', placeholder='ID pacjenta', description='ID:'),
    p_comment=Text(value='', placeholder='Uwagi...', description='Komentarz:')
);

# Tomograf


## Generowanie sinogramu i obrazu wyjściowego bez pokazania kroków pośrednich


In [ ]:
sinogram = scan_generate_sinogram(N_detectors, distance_between_emiters, angle_coverage, N_scans, img_matrix)
filtered_sinogram = filter_sinogram(sinogram)
reconstructed_image_filtered_matrix = reconstruct_image(angle_coverage, img_matrix, N_detectors, distance_between_emiters, N_scans, filtered_sinogram, rescale = True)

show_sinogram(sinogram)
# show_image(reconstructed_image_filtered_matrix)
dicom_file_path = save_reconstruction_to_dicom(reconstructed_image_filtered_matrix, patient_name=patient_name, patient_id=patient_id, comment=comment)
load_dicom_file(dicom_file_path)

## Generowanie sinogramu i obrazu wyjściowego z krokami pośrednimi


In [ ]:
from utils.interactive_tomograph import interactive_tomograph 

interactive_tomograph(img_matrix, N_detectors, distance_between_emiters, angle_coverage,  N_scans)

## Tworzenie Singoramu


In [ ]:
sinogram = scan_generate_sinogram(N_detectors, distance_between_emiters, angle_coverage, N_scans, img_matrix)

show_sinogram(sinogram)

## Filtr


In [ ]:
filtered_sinogram = filter_sinogram(sinogram)

show_sinogram(filtered_sinogram)

## Obraz wyjściowy (bez filtra)


In [ ]:
reconstructed_image_NOfilter_matrix = reconstruct_image(angle_coverage, img_matrix, N_detectors, distance_between_emiters, N_scans, sinogram, rescale= False)

show_image(reconstructed_image_NOfilter_matrix)

## Obraz wyjściowy (z filtrem)


In [ ]:
reconstructed_image_filtered_matrix = reconstruct_image(angle_coverage, img_matrix, N_detectors, distance_between_emiters, N_scans, filtered_sinogram, rescale = True)

show_image(reconstructed_image_filtered_matrix)